# 02. Data Cleaning and Imputation

Notebook này làm sạch dữ liệu cho bài toán **News Topic Classification**. Bộ dữ liệu đầu ra chính được thiết kế ở mức **dataset sạch dùng chung**, gồm bốn cột:

- `title`
- `description`
- `content`
- `category`

Ba cột `title`, `description`, `content` là văn bản đầu vào đã được làm sạch ở mức kỹ thuật; `category` là nhãn chủ đề lớn đã được chuẩn hóa. Các cột như `sub_topic`, `tag`, `source`, `url`, `author`, `public_date` được giữ trong file mở rộng/metadata để truy vết và phân tích phụ, nhưng không đưa vào file dữ liệu chính dùng cho bài toán phân loại.

Quy trình được thiết kế theo kết quả EDA: chuẩn hóa kỹ thuật ban đầu, xử lý missing values, label conflict, duplicate, bài quá ngắn, nhiễu văn bản, kiểm chứng lại và chia `train/dev/test` bằng stratified split.

Notebook này **không thực hiện** các bước phụ thuộc vào mô hình như xóa stopwords, tách từ, tokenize, encode hoặc vectorize. Những bước đó được để lại cho notebook modeling hoặc người sử dụng dataset tự quyết định theo mô hình cụ thể như TF-IDF, SVM, PhoBERT hoặc các Transformer khác.


## 0. Import và load dữ liệu

Dữ liệu được load từ ba nguồn báo: Tuổi Trẻ, Thanh Niên và VietnamNet. Sau khi load, mỗi dataframe được thêm cột `source` để lưu lại nguồn báo gốc, sau đó gộp thành một dataset chung.

**Lý do giữ `source` ở giai đoạn này:** `source` không phải nhãn cần dự đoán, nhưng cần dùng để kiểm tra source bias, truy vết dữ liệu và kiểm chứng việc cleaning có làm lệch tỷ trọng giữa các nguồn báo hay không. Cột này sẽ không được đưa vào file core dùng cho mô hình phân loại chủ đề.


In [1]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_PATHS = {
    'Tuổi Trẻ': '../data/rawdata/merged_data_tuoitre.csv',
    'Thanh Niên': '../data/rawdata/merged_data_thanhnien.csv',
    'VietnamNet': '../data/rawdata/merged_data_vietnamnet.csv'
}

dataframes = []

for source_name, path in DATA_PATHS.items():
    temp_df = pd.read_csv(path)
    temp_df['source'] = source_name
    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)

print(f'Shape ban đầu: {df.shape}')
print(f'Số dòng ban đầu: {df.shape[0]:,}')
print(f'Số cột ban đầu: {df.shape[1]:,}')
print('Danh sách cột:')
print(df.columns.tolist())

display(df.head())

Shape ban đầu: (273058, 10)
Số dòng ban đầu: 273,058
Số cột ban đầu: 10
Danh sách cột:
['category', 'sub_topic', 'title', 'description', 'content', 'public_date', 'author', 'source', 'url', 'tag']


,category,sub_topic,title,description,content,public_date,author,source,url,tag
0,Bất động sản,hoi đap,"Hoàn thiện chính sách đặc thù về đặc khu Vân Đồn, Vân Phong, Phú Quốc để trình Bộ Chính trị","Phó thủ tướng yêu cầu hoàn thiện đề án tổng thể về các cơ chế, chính sách đặc thù về phát triển các đặc khu Vân Đồn, Vân Phong và Phú Quốc, báo cáo Đảng ủy Chính phủ để trình Bộ Chính trị.",Đặc khu Vân Đồn - Ảnh: NGUYỄN TRIỀU\nVăn phòng Chính phủ vừa có thông báo kết luận của Phó thủ tướng thường trực Chính phủ Nguyễn Hòa Bình tại buổi làm việc với lãnh đạotỉnh Quảng Ninhđã nêu ra yê...,2025-12-30T19:31:59+07:00,NGỌC AN,Tuổi Trẻ,https://tuoitre.vn/hoan-thien-chinh-sach-dac-thu-ve-dac-khu-van-don-van-phong-phu-quoc-de-trinh-bo-chinh-tri-20251230185746359.htm,"vân đồn,vân phong,phú quốc,đặc khu vân đồn,đặc khu"
1,Bất động sản,hoi đap,"Các luật mới từ ngày 1-1 tác động đến đầu tư, đất đai, năng lượng","Từ ngày 1-1-2026, nhiều luật, nhóm chính sách quan trọng liên quan trực tiếp, có tác động lớn đến hoạt động đầu tư, phát triển đô thị, khai thác tài nguyên… sẽ có hiệu lực.","Từ ngày 1-1-2026, nhiều luật, nhóm chính sách quan trọng liên quan trực tiếp, có tác động lớn đến hoạt động đầu tư, phát triển đô thị, khai thác tài nguyên, định hướng chuyển dịch năng lượng… tron...",2025-12-30T18:43:19+07:00,CÔNG TRIỆU,Tuổi Trẻ,https://tuoitre.vn/cac-luat-moi-tu-ngay-1-1-tac-dong-den-dau-tu-dat-dai-nang-luong-20251230175453209.htm,"quy hoạch đô thị và nông thôn,lĩnh vực nông nghiệp,sử dụng năng lượng,năng lượng nguyên tử,luật đất đai"
2,Bất động sản,hoi đap,Thanh tra Chính phủ kết luận gì về dự án Golden Hills Đà Nẵng của Trung Nam?,"Thanh tra Chính phủ vừa có kết luận thanh tra các dự án có khó khăn, vướng mắc tại Đà Nẵng, trong đó có dự án Golden Hills City (hơn 342ha, chủ đầu tư là Công ty cổ phần đầu tư xây dựng Trung Nam).",Một góc dự án Golden Hills City - Ảnh: ĐOÀN CƯỜNG\nKết quả thanh tra cho thấy UBND TP Đà Nẵng không tổ chức đấu thầu lựa chọn nhà đầu tư thực hiệndự áncó sử dụng đất là vi phạm quy định tại khoản ...,2025-12-26T19:23:09+07:00,ĐOÀN CƯỜNG,Tuổi Trẻ,https://tuoitre.vn/thanh-tra-chinh-phu-ket-luan-gi-ve-du-an-golden-hills-da-nang-cua-trung-nam-2025122618582608.htm,"ubnd tp đà nẵng,golden hills,dự án khu đô thị,công ty trung nam,thanh tra"
3,Bất động sản,hoi đap,TP.HCM tính bồi thường hơn 156.000 tỉ đồng di dời 20.000 nhà ven kênh rạch đến 2030,"Theo kế hoạch do Sở Xây dựng TP.HCM trình, cần bồi thường hơn 156.000 tỉ đồng để di dời 20.000 nhà ven kênh rạch đến 2030 theo nghị quyết Đại hội Đảng bộ TP.HCM lần thứ nhất đặt ra.","TP.HCM sẽ di dời 20.000 nhà ven kênh rạch đến năm 2030 - Ảnh: CHÂU TUẤN\nSở Xây dựng vừa trình UBND TP.HCM về kế hoạch tổ chức thực hiện di dời nhà ven kênh rạch gồm đề án chỉnh trang đô thị, khu ...",2025-12-26T19:13:49+07:00,Ái Nhân,Tuổi Trẻ,https://tuoitre.vn/tp-hcm-tinh-boi-thuong-hon-156-000-ti-dong-di-doi-20-000-nha-ven-kenh-rach-den-2030-20251226185714376.htm,"nhà ven kênh rạch,di dời,bồi thường giải phóng mặt bằng,chỉnh trang đô thị,tp.hcm"
4,Bất động sản,hoi đap,HĐND TP.HCM thông qua thêm 64 khu đất thí điểm nhận đất nông nghiệp làm nhà ở thương mại,HĐND TP.HCM đã thông qua tiếp danh mục 76 khu đất thí điểm nhận đất nông nghiệp làm dự án nhà ở thương mại.,"Các khu đất làm dự án thí điểm nhà ở thương mại giúp TP.HCM sẽ có thêm nguồn cung nhà ở - Ảnh: TỰ TRUNG\nChiều 26-12, tại kỳ họp thứ 7 HĐND TP.HCM đã thông qua danh mục 64 khu đất dự kiến thực hiệ...",2025-12-26T19:09:21+07:00,Ái Nhân,Tuổi Trẻ,https://tuoitre.vn/hdnd-tp-hcm-thong-qua-them-64-khu-dat-thi-diem-nhan-dat-nong-nghiep-lam-nha-o-thuong-mai-20251226154756383.htm,"hđnd tp.hcm,khu đất,thí điểm thực hiện dự án nhà ở thương mại,đất nông nghiệp,nhà ở thương mại"


**Nhận xét:** Sau khi gộp ba nguồn báo, bảng shape và danh sách cột giúp kiểm tra nhanh dữ liệu đã đúng schema mong muốn hay chưa. Cột `source` được thêm vào để theo dõi nguồn gốc bài viết và phục vụ các bước kiểm tra source bias sau cleaning. Đây là dữ liệu đầu vào thô, chưa dùng trực tiếp để huấn luyện mô hình.

## 1. Chuẩn hóa kỹ thuật ban đầu

Trước khi xử lý missing, duplicate và label conflict, cần chuẩn hóa kỹ thuật ban đầu vì dữ liệu thu thập từ nhiều nguồn báo có thể chứa khoảng trắng thừa, nhiều khoảng trắng liên tiếp, ký tự xuống dòng/tab, chuỗi rỗng hoặc chuỗi toàn khoảng trắng. Ngoài ra, `category` có thể chưa thống nhất dấu/chữ hoa và `public_date` có thể chưa thống nhất kiểu dữ liệu.

Nếu không chuẩn hóa trước, một số dòng có vẻ không missing nhưng thực chất chỉ là khoảng trắng. Tương tự, hai bài giống nhau nhưng khác xuống dòng hoặc khoảng trắng có thể không được nhận diện là duplicate. Vì vậy, bước chuẩn hóa kỹ thuật được thực hiện trước các bước làm sạch chính thức.

In [2]:
rows_initial = len(df)

df_clean = df.copy()

string_cols = [
    'title',
    'description',
    'content',
    'sub_topic',
    'author',
    'tag',
    'category',
    'source',
    'url'
]
string_cols = [col for col in string_cols if col in df_clean.columns]

def normalize_basic_string(value):
    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize('NFC', str(value))
    value = re.sub(r'[\n\r\t]+', ' ', value)
    value = re.sub(r'\s+', ' ', value)
    value = value.strip()

    if value == '':
        return pd.NA

    return value

for col in string_cols:
    df_clean[col] = df_clean[col].map(normalize_basic_string)

CATEGORY_MAP = {
    'thoi su': 'Thời sự',
    'the gioi': 'Thế giới',
    'kinh doanh': 'Kinh doanh',
    'khoa hoc cong nghe': 'Khoa học công nghệ',
    'cong nghe': 'Khoa học công nghệ',
    'bat dong san': 'Bất động sản',
    'suc khoe': 'Sức khỏe',
    'giai tri': 'Giải trí',
    'the thao': 'Thể thao',
    'giao duc': 'Giáo dục',
    'phap luat': 'Pháp luật',
    'doi song': 'Đời sống',
    'xe': 'Xe',
    'du lich': 'Du lịch'
}

def remove_accents(text):
    if pd.isna(text):
        return pd.NA

    text = unicodedata.normalize('NFD', str(text))
    text = ''.join(char for char in text if unicodedata.category(char) != 'Mn')
    text = text.replace('đ', 'd').replace('Đ', 'D')
    return text

if 'category' in df_clean.columns:
    category_key = (
        df_clean['category']
        .map(remove_accents)
        .astype('string')
        .str.lower()
        .str.strip()
    )
    df_clean['category'] = category_key.map(CATEGORY_MAP).fillna(df_clean['category'])

if 'public_date' in df_clean.columns:
    df_clean['public_date'] = pd.to_datetime(df_clean['public_date'], errors='coerce')
    df_clean['year'] = df_clean['public_date'].dt.year.astype('Int64')
    df_clean['month'] = df_clean['public_date'].dt.month.astype('Int64')

rows_after_technical_normalization = len(df_clean)

print(f'Số dòng trước chuẩn hóa kỹ thuật: {rows_initial:,}')
print(f'Số dòng sau chuẩn hóa kỹ thuật: {rows_after_technical_normalization:,}')

if 'category' in df_clean.columns:
    print('Category sau chuẩn hóa:')
    display(pd.Series(sorted(df_clean['category'].dropna().unique()), name='category'))

missing_after_technical_normalization = (
    df_clean.isna().sum()
    .reset_index()
)
missing_after_technical_normalization.columns = ['column', 'missing_count']
missing_after_technical_normalization['missing_rate_percent'] = (
    missing_after_technical_normalization['missing_count'] / len(df_clean) * 100
).round(4)

missing_after_technical_normalization.sort_values('missing_count', ascending=False)

Số dòng trước chuẩn hóa kỹ thuật: 273,058
Số dòng sau chuẩn hóa kỹ thuật: 273,058
Category sau chuẩn hóa:


0           Bất động sản
1                Du lịch
2               Giáo dục
3               Giải trí
4     Khoa học công nghệ
5             Kinh doanh
6              Pháp luật
7               Sức khỏe
8               Thế giới
9               Thể thao
10               Thời sự
11                    Xe
12              Đời sống
Name: category, dtype: object

,column,missing_count,missing_rate_percent
11,month,124037,45.4251
10,year,124037,45.4251
5,public_date,124037,45.4251
6,author,27100,9.9246
4,content,2107,0.7716
3,description,158,0.0579
9,tag,85,0.0311
2,title,2,0.0007
0,category,0,0.0000
1,sub_topic,0,0.0000


**Nhận xét:** Bước này chỉ chuẩn hóa kỹ thuật, không nhằm loại bỏ bản ghi. Kết quả missing sau chuẩn hóa có thể tăng nhẹ vì các chuỗi rỗng hoặc chuỗi chỉ chứa khoảng trắng được chuyển về `pd.NA`. Đây là điều cần thiết để các bước missing, duplicate và label conflict phía sau phản ánh đúng hơn chất lượng dữ liệu.

## 2. Làm sạch dữ liệu theo quyết định từ EDA

Các bước làm sạch bên dưới chỉ xử lý những vấn đề đã được EDA chỉ ra: thiếu trường văn bản chính, label conflict, duplicate nội dung và bài quá ngắn.

### 2.1 Xử lý missing values

Theo EDA, `title`, `description` và `content` là ba trường văn bản chính để tạo `full_text`. Các dòng thiếu ít nhất một trong ba trường này không đủ thông tin đầu vào cho bài toán phân loại chủ đề, nên được loại bỏ.

Không fill `title`, `description` hoặc `content` vì việc tự điền text giả có thể tạo nhiễu ngữ nghĩa cho mô hình. Các cột như `author`, `tag`, `public_date` hoặc `year` không phải feature chính của mô hình trong notebook này, nên không dùng làm điều kiện drop ở bước này.

In [3]:
rows_before_missing = len(df_clean)

core_text_cols = [col for col in ['title', 'description', 'content'] if col in df_clean.columns]

missing_before = pd.DataFrame({
    'column': core_text_cols,
    'missing_count': [int(df_clean[col].isna().sum()) for col in core_text_cols],
    'missing_rate_percent': [round(df_clean[col].isna().mean() * 100, 4) for col in core_text_cols]
})

display(missing_before)

missing_core_mask = df_clean[core_text_cols].isna().any(axis=1)
missing_core_removed = int(missing_core_mask.sum())

df_clean = df_clean.loc[~missing_core_mask].copy()

rows_after_missing = len(df_clean)

missing_summary = pd.DataFrame({
    'metric': [
        'Rows before removing missing core text',
        'Rows removed due to missing title/description/content',
        'Rows after removing missing core text'
    ],
    'value': [
        rows_before_missing,
        missing_core_removed,
        rows_after_missing
    ]
})

display(missing_summary)

missing_core_after = pd.DataFrame({
    'column': core_text_cols,
    'missing_count_after': [int(df_clean[col].isna().sum()) for col in core_text_cols]
})

missing_core_after

,column,missing_count,missing_rate_percent
0,title,2,0.0007
1,description,158,0.0579
2,content,2107,0.7716


,metric,value
0,Rows before removing missing core text,273058
1,Rows removed due to missing title/description/content,2264
2,Rows after removing missing core text,270794


,column,missing_count_after
0,title,0
1,description,0
2,content,0


**Nhận xét:** Sau bước này, ba cột văn bản chính `title`, `description`, `content` phải không còn missing. Các cột phụ như `author`, `tag`, `public_date` hoặc `year` có thể vẫn thiếu vì chúng không phải đầu vào chính của mô hình phân loại chủ đề trong notebook này.

### 2.2 Xử lý label conflict

EDA phát hiện có trường hợp cùng một nội dung nhưng được gán `category` khác nhau. Đây là lỗi nghiêm trọng với bài toán supervised classification vì cùng một input nhưng có nhiều output khác nhau. Nếu giữ lại, mô hình sẽ học nhiễu nhãn và khó tối ưu.

Vì vậy, notebook loại bỏ toàn bộ các nhóm conflict, không chỉ giữ một dòng đại diện.

In [4]:
rows_before_label_conflict = len(df_clean)

conflict_subset = [col for col in ['title', 'description', 'content'] if col in df_clean.columns]

label_conflict_group = (
    df_clean.groupby(conflict_subset, dropna=False)
    .agg(
        n_rows=('category', 'size'),
        n_categories=('category', 'nunique'),
        categories=('category', lambda x: sorted(x.dropna().unique()))
    )
    .reset_index()
)

label_conflict_keys = label_conflict_group[label_conflict_group['n_categories'] > 1][conflict_subset]

if len(label_conflict_keys) > 0:
    conflict_marker = label_conflict_keys.copy()
    conflict_marker['_label_conflict'] = True
    conflict_mask = (
        df_clean.merge(conflict_marker, on=conflict_subset, how='left')['_label_conflict']
        .fillna(False)
        .to_numpy()
    )
else:
    conflict_mask = np.zeros(len(df_clean), dtype=bool)

df_label_conflict = df_clean.loc[conflict_mask].copy()

n_conflict_groups = int(len(label_conflict_keys))
label_conflict_removed = int(conflict_mask.sum())

label_conflict_summary = pd.DataFrame({
    'metric': [
        'Rows before removing label conflict',
        'Conflict groups',
        'Rows removed due to label conflict'
    ],
    'value': [
        rows_before_label_conflict,
        n_conflict_groups,
        label_conflict_removed
    ]
})

display(label_conflict_summary)

if label_conflict_removed > 0:
    display(
        df_label_conflict[
            [col for col in ['source', 'category', 'title', 'description', 'content', 'url'] if col in df_label_conflict.columns]
        ].head(10)
    )

df_clean = df_clean.loc[~conflict_mask].copy()
rows_after_label_conflict = len(df_clean)

print(f'Số dòng sau xử lý label conflict: {rows_after_label_conflict:,}')

label_conflict_group_after = (
    df_clean.groupby(conflict_subset, dropna=False)['category']
    .nunique()
    .reset_index(name='n_categories')
)

remaining_conflict_groups = int((label_conflict_group_after['n_categories'] > 1).sum())
print(f'Số nhóm label conflict sau xử lý: {remaining_conflict_groups:,}')

/tmp/ipykernel_57/836055894.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,metric,value
0,Rows before removing label conflict,270794
1,Conflict groups,15
2,Rows removed due to label conflict,30


,source,category,title,description,content,url
7871,Tuổi Trẻ,Giải trí,Liệu Taylor Swift có giành chiến thắng cho bà Kamala Harris?,"Taylor Swift, một trong những biểu tượng văn hóa đại chúng nổi tiếng nhất nước Mỹ và ước tính có đến hàng tỉ fan trên toàn thế giới vừa lên tiếng ủng hộ Phó tổng thống Kamala Harris trong cuộc đua...","Bài đăng của Taylor Swift trên tài khoản Instagram với hơn 283 triệu người theo dõi, thu hút gần 10 triệu lượt thích - Ảnh: Instagram nhân vật Bức tâm thư của Taylor Swiftđăng tải trên Instagram c...",https://tuoitre.vn/lieu-taylor-swift-co-gianh-chien-thang-cho-ba-kamala-harris-20240912084332993.htm
7895,Tuổi Trẻ,Giải trí,Universal Music yêu cầu chiến dịch vận động tranh cử của ông Trump phải gỡ nhạc ABBA,"Universal Music - hãng thu âm của ban nhạc Thụy Điển ABBA, ngày 29-8 đã ra thông báo yêu cầu ông Donald Trump ngừng sử dụng âm nhạc và video của nhóm nhạc này trong các cuộc vận động tranh cử.",Ông Trump và ban nhạc ABBA - Ảnh: CBC Chiến dịch vận động tranh cử của ông Trump và người liên danh JD Vance tại bang Minnesota hôm 27-7 có sử dụng nhiều bản hit của ABBA nhưThe Winner Takes It Al...,https://tuoitre.vn/universal-music-yeu-cau-chien-dich-van-dong-tranh-cu-cua-ong-trump-phai-go-nhac-abba-20240829204708295.htm
7951,Tuổi Trẻ,Giải trí,Ca khúc Freedom của Beyoncé được sử dụng để vận động tranh cử cho bà Kamala Harris,"Phó tổng thống Kamala Harris đã có màn ra mắt ấn tượng tại trụ sở chiến dịch tranh cử tổng thống của bà ở Wilmington, khi bước ra với ca khúc Freedom của Beyoncé.","Beyoncé đồng ý để bà Kamala Harris sử dụng ca khúc Freedom để vận động tranh cử - Ảnh: AFP/Getty Images Theo thông tin từ CNN, ""ong chúa"" Beyoncé đã đồng ý để Phó tổng thốngKamala Harrissử dụng ca...",https://tuoitre.vn/ca-khuc-freedom-cua-beyonce-duoc-su-dung-de-van-dong-tranh-cu-cho-ba-kamala-harris-20240724190149195.htm
10264,Tuổi Trẻ,Giải trí,"Ông Donald Trump thích Cuốn theo chiều gió, bà Kamala Harris mê vũ trụ Marvel?","Trong khi ông Donald Trump thích Cuốn theo chiều gió, Công dân Kane... thì bà Kamala Harris lại dành tình cảm đặc biệt cho Vũ trụ điện ảnh Marvel và phim về hệ thống hành pháp Mỹ.",Phim ảnh là một cách hay để đào sâu vào góc nhìn của ông Trump và bà Harris trước thềm bầu cử Mỹ - Ảnh: GETTY IMAGES Hai ứng cử viên tổng thống Mỹ là bà Kamala Harris và ôngDonald Trumpđang chạy đ...,https://tuoitre.vn/ong-donald-trump-thich-cuon-theo-chieu-gio-ba-kamala-harris-me-vu-tru-marvel-20241105131013558.htm
10430,Tuổi Trẻ,Giải trí,Những phim xuất sắc nhất để xem mùa bầu cử Mỹ,"Không chỉ đơn giản là một sự kiện chính trị quyết định tương lai đất nước, xuyên suốt lịch sử, cuộc bầu cử Mỹ ảnh hưởng đến tất cả mọi mặt đời sống như truyền thông, nghệ thuật, đặc biệt là điện ảnh.","Hình ảnh trong phim The Candidate, một trong những phim hài xuất sắc để xem mùa bầu cử Mỹ - Ảnh: IMDb Cũng từ đó, Hollywood sản sinh ra một thể loại phim riêng biệt để dành cho dịp đặc biệt này -p...",https://tuoitre.vn/nhung-phim-xuat-sac-nhat-de-xem-mua-bau-cu-my-20240826114949951.htm
15494,Tuổi Trẻ,Giải trí,Cardi B và sao Hollywood phản ứng trước chiến thắng của ông Trump: 'Tạm biệt nước Mỹ',"Cardi B, Ariana Grande và nhiều sao Hollywood bày tỏ phản ứng về tin ông Trump tái đắc cử, Billie Eilish gọi chiến thắng là 'Cuộc chiến chống lại phụ nữ'.","Ông Trump TheoThe Independent, nhiều người ủng hộ Đảng Dân chủ và sao Hollywood đang bày tỏ lo lắng về tương lai trước sự kiện ôngDonald Trumptái đắc cử tổng thống thứ 47 của Mỹ hôm 6-11. Tuy nhiê...",https://tuoitre.vn/cardi-b-va-sao-hollywood-phan-ung-truoc-chien-thang-cua-ong-trump-tam-biet-nuoc-my-20241107102027523.htm
15496,Tuổi Trẻ,Giải trí,"Người ủng hộ ông Trump ăn mừng náo nhiệt trên X, các siêu sao phe bà Harris im lặng","Sau tuyên bố chiến thắng của ông Donald Trump tại Trung tâm hội nghị bang Florida (Mỹ), những người nổi tiếng từng lên tiếng ủng hộ vị tân tổng thống Mỹ chia sẻ niềm vui của mình trên mạng xã hội.","Ông Elon Musk ăn mừng bằng ảnh vui ông đang cầm một cái bồn rửa tay

Số dòng sau xử lý label conflict: 270,764
Số nhóm label conflict sau xử lý: 0


**Nhận xét:** Các nhóm label conflict bị loại bỏ hoàn toàn vì cùng một bộ văn bản nhưng có nhiều nhãn `category` khác nhau sẽ làm mô hình học tín hiệu mâu thuẫn. Sau xử lý, số nhóm conflict kiểm chứng lại phải bằng 0.

### 2.3 Xử lý duplicate

Theo EDA, dataset không có trùng toàn bộ dòng và không có URL trùng nguyên văn, nhưng vẫn có duplicate theo các trường văn bản. Không drop theo `title` riêng lẻ vì một số tiêu đề có thể lặp lại tự nhiên ở các tin định kỳ. Không drop theo `description` riêng lẻ vì mô tả có thể ngắn hoặc mang tính template.

Với bài toán dùng `title + description + content`, `content` là phần dài nhất và chiếm nhiều tín hiệu nhất. Do đó, duplicate content có khả năng tạo bias mạnh nhất. Nhóm trùng cả `title + description + content` cũng nằm trong nhóm duplicate content, nên chỉ cần deduplicate theo `content` để tránh xử lý trùng lặp hai lần.

In [5]:
rows_before_content_dedup = len(df_clean)

duplicate_before_rows = []

duplicate_before_rows.append({
    'duplicate_type': 'full row',
    'duplicate_count': int(df_clean.duplicated().sum())
})

if 'url' in df_clean.columns:
    duplicate_before_rows.append({
        'duplicate_type': 'url',
        'duplicate_count': int(df_clean.duplicated(subset=['url']).sum())
    })

for col in ['title', 'description', 'content']:
    if col in df_clean.columns:
        duplicate_before_rows.append({
            'duplicate_type': col,
            'duplicate_count': int(df_clean.duplicated(subset=[col]).sum())
        })

all_text_subset = [col for col in ['title', 'description', 'content'] if col in df_clean.columns]

duplicate_before_rows.append({
    'duplicate_type': 'title + description + content',
    'duplicate_count': int(df_clean.duplicated(subset=all_text_subset).sum())
})

duplicate_before_summary = pd.DataFrame(duplicate_before_rows)
display(duplicate_before_summary)

content_dup_group_stats = (
    df_clean.groupby('content', dropna=False)
    .size()
    .reset_index(name='n_rows')
)

duplicate_content_groups = int((content_dup_group_stats['n_rows'] > 1).sum())
rows_in_duplicate_content_groups = int(
    content_dup_group_stats.loc[content_dup_group_stats['n_rows'] > 1, 'n_rows'].sum()
)
duplicate_content_removed = int(df_clean.duplicated(subset=['content']).sum())

df_clean = (
    df_clean
    .drop_duplicates(subset=['content'], keep='first')
    .reset_index(drop=True)
)

rows_after_content_dedup = len(df_clean)

content_dedup_summary = pd.DataFrame({
    'metric': [
        'Rows before content deduplication',
        'Duplicate content groups',
        'Rows in duplicate content groups',
        'Rows removed due to duplicate content',
        'Rows after content deduplication'
    ],
    'value': [
        rows_before_content_dedup,
        duplicate_content_groups,
        rows_in_duplicate_content_groups,
        duplicate_content_removed,
        rows_after_content_dedup
    ]
})

display(content_dedup_summary)

content_duplicate_after = int(df_clean.duplicated(subset=['content']).sum())
all3_duplicate_after = int(df_clean.duplicated(subset=all_text_subset).sum())

print(f'Số duplicate content sau xử lý: {content_duplicate_after:,}')
print(f'Số duplicate title + description + content sau xử lý: {all3_duplicate_after:,}')

,duplicate_type,duplicate_count
0,full row,0
1,url,0
2,title,630
3,description,1052
4,content,1999
5,title + description + content,81


,metric,value
0,Rows before content deduplication,270764
1,Duplicate content groups,378
2,Rows in duplicate content groups,2377
3,Rows removed due to duplicate content,1999
4,Rows after content deduplication,268765


Số duplicate content sau xử lý: 0
Số duplicate title + description + content sau xử lý: 0


**Nhận xét:** Deduplicate được thực hiện theo `content` vì đây là phần chiếm nhiều tín hiệu nhất trong `full_text`. Sau bước này, duplicate theo `content` và duplicate theo cả `title + description + content` phải bằng 0. Duplicate theo `title` hoặc `description` riêng lẻ không bắt buộc bằng 0 vì có thể là các bài định kỳ hoặc mô tả dạng template hợp lệ.

### 2.4 Xử lý độ dài văn bản

Trước khi xử lý độ dài, tạo `full_text` tạm thời bằng cách nối `title`, `description` và `content`. Theo EDA, `full_text` có median khoảng 649 từ, mean khoảng 719 từ, P01 khoảng 49 từ và P99 khoảng 1,945 từ.

Các bài quá ngắn với `full_text_wc <= 49` có khả năng crawl thiếu nội dung hoặc không đủ ngữ cảnh để phân loại, nên được drop tự động.

Ngược lại, các bài quá dài với `full_text_wc >= 1,945` không được drop tự động. Kết quả kiểm tra thủ công cho thấy phần lớn các bài này vẫn là bài báo hợp lệ như longform, bài phân tích chuyên sâu, bài tổng hợp sự kiện hoặc bài tư vấn dài. Nếu loại bỏ chỉ vì vượt ngưỡng P99, dữ liệu có thể mất nhiều bài viết giàu ngữ cảnh và có giá trị cho mô hình.

Tuy nhiên, quá trình kiểm tra thủ công cũng cho thấy một số bài dài có chứa nhiễu do crawl như URL nhúng, chú thích ảnh dạng `ẢNH:`/`Ảnh:`, text điều hướng video, banner quảng cáo hoặc dòng dẫn sang bài khác. Vì vậy, notebook này không drop bài dài, mà xử lý các thành phần nhiễu này trong bước chuẩn hóa văn bản bằng regex.

Không drop title ngắn hoặc description ngắn riêng lẻ, vì bản chất của tiêu đề và mô tả là ngắn. Không drop content dài nếu bài vẫn hợp lệ.

In [6]:
rows_before_length_filter = len(df_clean)

df_clean['full_text'] = (
    df_clean['title'].astype(str)
    + '. '
    + df_clean['description'].astype(str)
    + '. '
    + df_clean['content'].astype(str)
)

df_clean['full_text_wc'] = (
    df_clean['full_text']
    .fillna('')
    .astype(str)
    .str.split()
    .str.len()
)

print('Thống kê full_text_wc trước xử lý độ dài:')
display(
    df_clean['full_text_wc']
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .to_frame('full_text_wc')
)

short_article_mask = df_clean['full_text_wc'] <= 49
short_articles_removed = int(short_article_mask.sum())

df_clean = df_clean.loc[~short_article_mask].copy()

rows_after_length_filter = len(df_clean)
long_articles_kept = int((df_clean['full_text_wc'] >= 1945).sum())

length_filter_summary = pd.DataFrame({
    'metric': [
        'Rows before length filtering',
        'Rows removed because full_text_wc <= 49',
        'Rows after length filtering',
        'Long articles kept because full_text_wc >= 1945'
    ],
    'value': [
        rows_before_length_filter,
        short_articles_removed,
        rows_after_length_filter,
        long_articles_kept
    ]
})

display(length_filter_summary)

if long_articles_kept > 0:
    display(
        df_clean.loc[
            df_clean['full_text_wc'] >= 1945,
            [col for col in ['source', 'category', 'title', 'full_text_wc', 'url'] if col in df_clean.columns]
        ].head(10)
    )

remaining_short_articles = int((df_clean['full_text_wc'] <= 49).sum())

print(f'Số bài full_text_wc <= 49 sau xử lý: {remaining_short_articles:,}')
print('Thống kê full_text_wc sau xử lý độ dài:')
display(
    df_clean['full_text_wc']
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .to_frame('full_text_wc')
)

Thống kê full_text_wc trước xử lý độ dài:


,full_text_wc
count,"268,765.0000"
mean,728.2888
std,401.8758
min,28.0000
1%,86.0000
5%,277.0000
50%,656.0000
95%,"1,429.0000"
99%,"1,946.0000"
max,"36,198.0000"


,metric,value
0,Rows before length filtering,268765
1,Rows removed because full_text_wc <= 49,534
2,Rows after length filtering,268231
3,Long articles kept because full_text_wc >= 1945,2700


,source,category,title,full_text_wc,url
1,Tuổi Trẻ,Bất động sản,"Các luật mới từ ngày 1-1 tác động đến đầu tư, đất đai, năng lượng",2024,https://tuoitre.vn/cac-luat-moi-tu-ngay-1-1-tac-dong-den-dau-tu-dat-dai-nang-luong-20251230175453209.htm
36,Tuổi Trẻ,Bất động sản,Đề xuất chuyển 300 ha đất vàng Khu chế xuất Tân Thuận thành Khu công nghệ cao,2759,https://tuoitre.vn/de-xuat-chuyen-300-ha-dat-vang-khu-che-xuat-tan-thuan-thanh-khu-cong-nghe-cao-20251205081317006.htm
75,Tuổi Trẻ,Bất động sản,Tiến tới bỏ công chứng khi mua bán nhà đất,2800,https://tuoitre.vn/tien-toi-bo-cong-chung-khi-mua-ban-nha-dat-20251017075645486.htm
85,Tuổi Trẻ,Bất động sản,"Làm nhà ở xã hội: Tại sao có nơi làm tốt, nơi không?",2709,https://tuoitre.vn/lam-nha-o-xa-hoi-tai-sao-co-noi-lam-tot-noi-khong-20251005081918425.htm
211,Tuổi Trẻ,Bất động sản,"Thủ tướng thúc địa phương làm nhà cho dân, chuyên gia nói nên 'đánh' vào thời gian phê duyệt đầu tư",2333,https://tuoitre.vn/thu-tuong-thuc-dia-phuong-lam-nha-cho-dan-chuyen-gia-noi-nen-danh-vao-thoi-gian-phe-duyet-dau-tu-20250302093255846.htm
296,Tuổi Trẻ,Bất động sản,"Từ 1-8, điểm nghẽn định giá đất được gỡ, loạt dự án ở TP.HCM sẽ sớm có sổ hồng",2543,https://tuoitre.vn/tu-1-8-diem-nghen-dinh-gia-dat-duoc-go-loat-du-an-o-tp-hcm-se-som-co-so-hong-20240705080753668.htm
343,Tuổi Trẻ,Bất động sản,Quản trị chung cư: Cần ngăn những cơn 'sóng ngầm' ở đô thị,3348,https://tuoitre.vn/quan-tri-chung-cu-can-ngan-nhung-con-song-ngam-o-do-thi-20240317084602874.htm
415,Tuổi Trẻ,Bất động sản,Phải bảo đảm quyền tách thửa của người dân,2851,https://tuoitre.vn/phai-bao-dam-quyen-tach-thua-cua-nguoi-dan-20231020085535722.htm
552,Tuổi Trẻ,Bất động sản,Chi phí nào khiến giá nhà cao bất thường?,2981,https://tuoitre.vn/chi-phi-nao-khien-gia-nha-cao-bat-thuong-20240923083409462.htm
590,Tuổi Trẻ,Bất động sản,"Giao dịch bất động sản tăng, chờ sôi động trở lại",2973,https://tuoitre.vn/giao-dich-bat-dong-san-tang-cho-soi-dong-tro-lai-20240623074751839.htm


Số bài full_text_wc <= 49 sau xử lý: 0
Thống kê full_text_wc sau xử lý độ dài:


,full_text_wc
count,"268,231.0000"
mean,729.6512
std,401.1127
min,50.0000
1%,99.0000
5%,281.0000
50%,657.0000
95%,"1,430.0000"
99%,"1,947.7000"
max,"36,198.0000"


**Nhận xét:** Sau khi drop các bài quá ngắn, giá trị nhỏ nhất của `full_text_wc` phải lớn hơn 49. Các bài rất dài vẫn được giữ lại vì kiểm tra thủ công cho thấy nhiều bài là longform, phân tích chuyên sâu hoặc tổng hợp sự kiện hợp lệ. Những nhiễu thường gặp trong bài dài như URL, chú thích ảnh hoặc dòng điều hướng sẽ được xử lý ở bước chuẩn hóa văn bản cho mô hình.

## 3. Chuẩn hóa văn bản ở mức dataset dùng chung

Sau khi đã xử lý missing values, duplicate, label conflict và bài quá ngắn, notebook chuẩn hóa trực tiếp ba cột văn bản lõi `title`, `description`, `content`. Đây là ba cột sẽ được lưu ra file dữ liệu chính.

Mục tiêu của bước này là tạo một bộ dữ liệu sạch nhưng vẫn đủ tổng quát để nhiều mô hình khác nhau có thể sử dụng. Vì vậy, bước này chỉ xử lý các nhiễu gần như chắc chắn là lỗi kỹ thuật hoặc rác do crawl:

- Chuẩn hóa Unicode về NFC để thống nhất cách biểu diễn ký tự tiếng Việt.
- Loại bỏ URL nhúng như `http://...`, `https://...`, `www...`.
- Loại bỏ cụm điều hướng như `Xem thêm`, `Đọc thêm`, `Xem chi tiết`.
- Loại bỏ chú thích ảnh hoặc nguồn ảnh như `ẢNH:`, `Ảnh:`, `Nguồn ảnh:`.
- Loại bỏ một số cụm điều khiển video nếu bị lẫn vào nội dung do crawl.
- Loại bỏ emoji, icon, symbol và các ký tự lỗi đã được EDA chỉ ra.
- Loại bỏ ký tự lỗi hiển thị như `�` và control characters.
- Chuẩn hóa nhiều khoảng trắng liên tiếp thành một khoảng trắng.
- Loại bỏ khoảng trắng thừa ở đầu và cuối văn bản.

**Các bước cố ý không làm ở notebook cleaning:**

- Không lowercase bắt buộc, vì chữ hoa/chữ thường có thể chứa thông tin về tên riêng, địa danh, tổ chức hoặc tiêu đề.
- Không xóa stopwords, vì stopwords có thể còn hữu ích với Transformer và một số biểu diễn ngữ cảnh.
- Không tách từ, tokenize, encode hoặc vectorize, vì các bước này phụ thuộc vào mô hình sử dụng sau này.
- Không xóa chữ số, vì tin tức chứa nhiều thông tin số quan trọng như ngày tháng, tỷ lệ, giá tiền, điểm số và số liệu kinh tế.
- Không xóa toàn bộ dấu câu quá mạnh, vì việc xử lý dấu câu nên phụ thuộc tokenizer hoặc vectorizer ở bước modeling.

Nói cách khác, notebook này chỉ làm sạch những vấn đề đã phát hiện trong EDA và chắc chắn là nhiễu dữ liệu. Các bước tiền xử lý mang tính lựa chọn sẽ được để cho người dùng dataset hoặc notebook modeling quyết định.


In [7]:
rows_before_final_text_cleaning = len(df_clean)

# Lưu lại full_text trước bước chuẩn hóa dùng chung để phục vụ kiểm tra trực quan trước/sau.
df_clean['full_text_before_shared_cleaning'] = (
    df_clean['title'].astype(str)
    + '. '
    + df_clean['description'].astype(str)
    + '. '
    + df_clean['content'].astype(str)
)

emoji_symbol_pattern = re.compile(
    '['
    '\U0001F000-\U0001FAFF'
    '\U00002600-\U000027BF'
    '\U00002190-\U000021FF'
    '\U00002300-\U000023FF'
    '\U000025A0-\U000025FF'
    '\U00002B00-\U00002BFF'
    '\U0001F800-\U0001F8FF'
    '\U0000FE0F'
    '\U0000200D'
    ']+' ,
    flags=re.UNICODE
)

extra_noise_chars = '™®©℠℃㎡℅㎛〒￼˚`^´¸˝°'
extra_noise_pattern = re.compile('[' + re.escape(extra_noise_chars) + ']')


def clean_text_for_shared_dataset(text):
    """Làm sạch văn bản ở mức dataset dùng chung.

    Hàm này chỉ xử lý nhiễu kỹ thuật/rác crawl. Hàm cố ý không lowercase,
    không xóa stopwords, không tách từ và không tokenize để không áp đặt
    giả định tiền xử lý lên các mô hình sử dụng dataset sau này.
    """
    if pd.isna(text):
        return pd.NA

    text = unicodedata.normalize('NFC', str(text))
    text = text.replace('�', ' ')

    # URL thường là rác crawl hoặc đường dẫn điều hướng, không phải nội dung ngôn ngữ chính.
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Chú thích ảnh/nguồn ảnh thường không phải nội dung chính của bài báo.
    text = re.sub(r'\b(ẢNH|Ảnh|Nguồn ảnh|Nguồn)\s*:\s*[^\.。\n\r]{0,120}', ' ', text)
    text = re.sub(r'\bTrong ảnh\s*:?', ' ', text, flags=re.IGNORECASE)

    # Các cụm điều hướng nội bộ của website nếu bị crawl lẫn vào văn bản.
    text = re.sub(r'\bXem thêm\s*:?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bĐọc thêm\s*:?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bXem chi tiết[^\.。\n\r]{0,150}', ' ', text, flags=re.IGNORECASE)

    # Một số cụm điều khiển video/interactive có thể bị lẫn vào nội dung do crawl.
    text = re.sub(
        r'\b(play\s+mute|mute\s+unmute|fullscreen\s+exit fullscreen|next\s+previous)\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    text = emoji_symbol_pattern.sub(' ', text)
    text = extra_noise_pattern.sub(' ', text)

    # Loại control characters nhưng giữ lại chữ, số, dấu câu thông thường và dấu tiếng Việt.
    text = ''.join(
        ' ' if unicodedata.category(char)[0] == 'C' else char
        for char in text
    )

    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    if text == '':
        return pd.NA

    return text


required_text_cols = ['title', 'description', 'content']
cleaned_text_cols = {}

for col in required_text_cols:
    clean_col = f'{col}_clean'
    df_clean[clean_col] = df_clean[col].map(clean_text_for_shared_dataset)
    cleaned_text_cols[col] = clean_col

# Tạo candidate full text từ ba cột đã làm sạch để kiểm tra rỗng/quá ngắn sau chuẩn hóa.
df_clean['full_text_clean_candidate'] = (
    df_clean['title_clean'].astype(str)
    + '. '
    + df_clean['description_clean'].astype(str)
    + '. '
    + df_clean['content_clean'].astype(str)
)

df_clean['full_text_wc_clean_candidate'] = (
    df_clean['full_text_clean_candidate']
    .fillna('')
    .astype(str)
    .str.split()
    .str.len()
)

empty_required_text_mask = pd.Series(False, index=df_clean.index)

for clean_col in cleaned_text_cols.values():
    empty_required_text_mask = (
        empty_required_text_mask
        | df_clean[clean_col].isna()
        | (df_clean[clean_col].fillna('').astype(str).str.strip() == '')
    )

invalid_clean_core_mask = (
    empty_required_text_mask
    | (df_clean['full_text_wc_clean_candidate'] <= 49)
)

invalid_clean_core_removed = int(invalid_clean_core_mask.sum())

if invalid_clean_core_removed > 0:
    df_clean = df_clean.loc[~invalid_clean_core_mask].copy()

# Ghi đè ba cột lõi bằng phiên bản đã làm sạch.
# Nhờ vậy file output chính vẫn chỉ có title, description, content, category.
for original_col, clean_col in cleaned_text_cols.items():
    df_clean[original_col] = df_clean[clean_col]

helper_cols_to_drop = list(cleaned_text_cols.values()) + [
    'full_text_clean_candidate',
    'full_text_wc_clean_candidate'
]
df_clean = df_clean.drop(columns=[col for col in helper_cols_to_drop if col in df_clean.columns])

# Tạo full_text_clean chỉ để kiểm chứng/audit và hỗ trợ notebook modeling nếu cần.
df_clean['full_text'] = (
    df_clean['title'].astype(str)
    + '. '
    + df_clean['description'].astype(str)
    + '. '
    + df_clean['content'].astype(str)
)
df_clean['full_text_clean'] = df_clean['full_text']

df_clean['full_text_wc_clean'] = (
    df_clean['full_text_clean']
    .fillna('')
    .astype(str)
    .str.split()
    .str.len()
)

# Sau khi chuẩn hóa văn bản, có thể phát sinh duplicate hoặc label conflict mới
# do URL/chú thích ảnh/ký tự nhiễu bị loại bỏ. Vì vậy cần kiểm tra lại một lần nữa.
clean_conflict_subset = ['title', 'description', 'content']
clean_label_conflict_group = (
    df_clean.groupby(clean_conflict_subset, dropna=False)
    .agg(n_categories=('category', 'nunique'))
    .reset_index()
)

clean_label_conflict_keys = clean_label_conflict_group.loc[
    clean_label_conflict_group['n_categories'] > 1,
    clean_conflict_subset
]

if len(clean_label_conflict_keys) > 0:
    clean_label_conflict_key_set = set(map(tuple, clean_label_conflict_keys.to_numpy()))
    post_clean_label_conflict_mask = df_clean[clean_conflict_subset].apply(tuple, axis=1).isin(
        clean_label_conflict_key_set
    )
else:
    post_clean_label_conflict_mask = pd.Series(False, index=df_clean.index)

post_clean_label_conflict_removed = int(post_clean_label_conflict_mask.sum())

if post_clean_label_conflict_removed > 0:
    df_clean = df_clean.loc[~post_clean_label_conflict_mask].copy()

post_clean_duplicate_content_removed = int(df_clean.duplicated(subset=['content']).sum())

if post_clean_duplicate_content_removed > 0:
    df_clean = df_clean.drop_duplicates(subset=['content'], keep='first').copy()

# Recompute sau khi có thể đã drop thêm conflict/duplicate phát sinh sau chuẩn hóa.
df_clean['full_text'] = (
    df_clean['title'].astype(str)
    + '. '
    + df_clean['description'].astype(str)
    + '. '
    + df_clean['content'].astype(str)
)
df_clean['full_text_clean'] = df_clean['full_text']
df_clean['full_text_wc_clean'] = (
    df_clean['full_text_clean']
    .fillna('')
    .astype(str)
    .str.split()
    .str.len()
)

# Đồng bộ full_text_wc với phiên bản văn bản cuối cùng để các thống kê sau không bị lệch.
df_clean['full_text_wc'] = df_clean['full_text_wc_clean']

rows_after_final_text_cleaning = len(df_clean)
final_text_cleaning_removed = (
    invalid_clean_core_removed
    + post_clean_label_conflict_removed
    + post_clean_duplicate_content_removed
)

if len(df_clean) > 0:
    min_wc_clean = int(df_clean['full_text_wc_clean'].min())
    median_wc_clean = float(df_clean['full_text_wc_clean'].median())
    max_wc_clean = int(df_clean['full_text_wc_clean'].max())
else:
    min_wc_clean = 0
    median_wc_clean = 0.0
    max_wc_clean = 0

text_clean_check = pd.DataFrame({
    'metric': [
        'Rows before shared text cleaning',
        'Rows removed because cleaned core text is empty or too short',
        'Rows removed because cleaning created label conflict',
        'Rows removed because cleaning created duplicate content',
        'Rows after shared text cleaning',
        'title missing after cleaning',
        'description missing after cleaning',
        'content missing after cleaning',
        'full_text_clean missing',
        'full_text_clean empty string',
        'Minimum full_text_wc_clean',
        'Median full_text_wc_clean',
        'Maximum full_text_wc_clean',
        'Long articles kept after shared text cleaning'
    ],
    'value': [
        rows_before_final_text_cleaning,
        invalid_clean_core_removed,
        post_clean_label_conflict_removed,
        post_clean_duplicate_content_removed,
        rows_after_final_text_cleaning,
        int(df_clean['title'].isna().sum()),
        int(df_clean['description'].isna().sum()),
        int(df_clean['content'].isna().sum()),
        int(df_clean['full_text_clean'].isna().sum()),
        int((df_clean['full_text_clean'].fillna('').str.strip() == '').sum()),
        min_wc_clean,
        median_wc_clean,
        max_wc_clean,
        int((df_clean['full_text_wc_clean'] >= 1945).sum())
    ]
})

display(text_clean_check)

print('Thống kê độ dài full_text_wc_clean:')
display(
    df_clean['full_text_wc_clean']
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)


,metric,value
0,Rows before shared text cleaning,"268,231.0000"
1,Rows removed because cleaned core text is empty or too short,101.0000
2,Rows removed because cleaning created label conflict,0.0000
3,Rows removed because cleaning created duplicate content,12.0000
4,Rows after shared text cleaning,"268,118.0000"
5,title missing after cleaning,0.0000
6,description missing after cleaning,0.0000
7,content missing after cleaning,0.0000
8,full_text_clean missing,0.0000
9,full_text_clean empty string,0.0000


Thống kê độ dài full_text_wc_clean:


count   268,118.0000
mean        713.1887
std         394.3896
min          50.0000
1%          100.0000
5%          277.0000
50%         638.0000
95%       1,402.0000
99%       1,920.8300
max      35,795.0000
Name: full_text_wc_clean, dtype: float64

**Nhận xét:** Bước này tạo phiên bản văn bản sạch ở mức dataset dùng chung bằng cách ghi đè trực tiếp ba cột lõi `title`, `description`, `content`. Cách này giúp file đầu ra chính vẫn giữ schema đơn giản, đúng mục tiêu bài toán và không bắt người dùng phải phụ thuộc vào cột trung gian như `full_text_clean`.

Notebook cố ý không lowercase, không xóa stopwords, không tách từ và không tokenize vì các bước đó phụ thuộc vào mô hình. Ví dụ, mô hình TF-IDF có thể cần pipeline khác với PhoBERT hoặc Transformer. Nếu làm các bước đó ngay trong dataset release, bộ dữ liệu sẽ kém linh hoạt và có thể làm mất thông tin mà một số mô hình cần.

Sau chuẩn hóa, notebook kiểm tra lại text rỗng/quá ngắn, label conflict và duplicate vì việc loại URL, chú thích ảnh hoặc ký tự nhiễu có thể làm một số bản ghi trước đó khác nhau trở nên giống nhau. Đây là bước phòng ngừa để file cuối không còn lỗi phát sinh sau cleaning.


In [8]:
before_clean_col = 'full_text_before_shared_cleaning' if 'full_text_before_shared_cleaning' in df_clean.columns else 'full_text'

noise_sample_mask = df_clean[before_clean_col].fillna('').astype(str).str.contains(
    r'https?://|www\.|ẢNH:|Ảnh:|Xem thêm|Xem chi tiết|Đọc thêm',
    regex=True,
    case=False
)

sample_n = min(3, int(noise_sample_mask.sum()))

if sample_n > 0:
    sample_rows = df_clean.loc[noise_sample_mask].sample(sample_n, random_state=42)
elif len(df_clean) > 0:
    sample_rows = df_clean.sample(min(3, len(df_clean)), random_state=42)
else:
    sample_rows = df_clean.head(0)

for idx, row in sample_rows.iterrows():
    print('=' * 120)
    print(f'Index: {idx}')
    print(f"Source: {row.get('source', '')}")
    print(f"Category: {row.get('category', '')}")
    print(f"Title sau cleaning: {row.get('title', '')}")

    print('\n--- FULL TEXT TRƯỚC CHUẨN HÓA DÙNG CHUNG ---')
    print(str(row[before_clean_col])[:2000])

    print('\n--- FULL TEXT SAU CHUẨN HÓA DÙNG CHUNG ---')
    print(str(row['full_text_clean'])[:2000])
    print()


Index: 16046
Source: Tuổi Trẻ
Category: Giải trí
Title sau cleaning: Tin tức giải trí 12-2: Gala cười được khen hơn Táo quân Tết Giáp Thìn

--- FULL TEXT TRƯỚC CHUẨN HÓA DÙNG CHUNG ---
Tin tức giải trí 12-2: Gala cười được khen hơn Táo quân Tết Giáp Thìn. Một số tin tức đáng chú ý: Khán giả khen Gala cười hay hơn Táo quân; Lãnh Thanh - 'nam thần trong mộng' của Ước mình cùng bay; 12 con giáp nói về vị xa quê.. Một tiết mục trong chương trình Gala cười 2024 - Ảnh: VTV Chương trìnhGala cườiphát sóng tối mùng 2 Tết (11-2) trên VTV3 chỉ có ba tiểu phẩm và không có sự tham gia của các nghệ sĩ miền Nam, tạo cảm giác hơi nghèo nàn. Dù vậy năm nay các tiết mục củaGala cườiphần lớn nhận được lời khen của khán giả. Trong đó, ca khúc nhạc chế được yêu thích, lan truyền khá mạnh trên TikTok. Một số khán giả xem chương trình còn so sánh giữaGala cườivàTáo quântrên VTV: "Gala cườikhông bị giới hạn bởi bối cảnh, chủ đề và mô típ cho nên không tạo cảm giác nhàm chán. Thái Sơn và Thái Dương rất xuất sắ

**Nhận xét:** Các mẫu so sánh trước và sau chuẩn hóa giúp kiểm tra trực quan xem URL, chú thích ảnh, dòng điều hướng, ký tự lỗi, emoji/icon và khoảng trắng thừa đã được xử lý đúng chưa. Điểm cần chú ý là văn bản sau cleaning vẫn giữ chữ hoa/chữ thường, chữ số, dấu câu cơ bản và dấu tiếng Việt để không áp đặt preprocessing theo một mô hình cụ thể.


## 4. Kiểm chứng lại toàn bộ vấn đề từ EDA sau cleaning

Sau khi hoàn thành các bước làm sạch và chuẩn hóa văn bản, cần kiểm chứng lại các vấn đề chính đã được phát hiện trong EDA: missing, duplicate, label conflict, độ dài văn bản, phân phối nhãn, phân phối source và nhiễu văn bản còn sót lại.

In [9]:
pipeline_summary = pd.DataFrame({
    'stage': [
        'Initial raw data',
        'After technical normalization',
        'After removing missing core text',
        'After removing label conflict',
        'After content deduplication',
        'After removing very short articles',
        'After shared text cleaning and re-validation',
        'Final cleaned data'
    ],
    'n_rows': [
        rows_initial,
        rows_after_technical_normalization,
        rows_after_missing,
        rows_after_label_conflict,
        rows_after_content_dedup,
        rows_after_length_filter,
        rows_after_final_text_cleaning,
        len(df_clean)
    ],
    'rows_removed_at_step': [
        0,
        rows_initial - rows_after_technical_normalization,
        missing_core_removed,
        label_conflict_removed,
        duplicate_content_removed,
        short_articles_removed,
        final_text_cleaning_removed,
        0
    ]
})

pipeline_summary['removed_rate_percent_at_step'] = (
    pipeline_summary['rows_removed_at_step'] / rows_initial * 100
).round(4)

pipeline_summary


,stage,n_rows,rows_removed_at_step,removed_rate_percent_at_step
0,Initial raw data,273058,0,0.0000
1,After technical normalization,273058,0,0.0000
2,After removing missing core text,270794,2264,0.8291
3,After removing label conflict,270764,30,0.0110
4,After content deduplication,268765,1999,0.7321
5,After removing very short articles,268231,534,0.1956
6,After shared text cleaning and re-validation,268118,113,0.0414
7,Final cleaned data,268118,0,0.0000


**Nhận xét:** Bảng tổng hợp này thay cho audit log. Nó cho biết số dòng còn lại sau từng bước, số dòng bị loại ở mỗi bước và tỷ lệ bị loại so với dữ liệu ban đầu. Nếu một bước làm giảm dữ liệu quá mạnh, cần quay lại kiểm tra nguyên nhân trước khi dùng dữ liệu cho modeling.

### 4.1 Kiểm chứng missing

Sau cleaning, ba cột text chính `title`, `description`, `content` và cột audit `full_text_clean` phải không còn missing hoặc chuỗi rỗng. Các cột phụ như `author`, `tag`, `sub_topic`, `public_date`, `year` có thể vẫn thiếu vì chúng không phải feature bắt buộc trong file dữ liệu chính.


In [10]:
missing_validation_cols = [col for col in ['title', 'description', 'content', 'full_text_clean'] if col in df_clean.columns]

missing_validation = pd.DataFrame({
    'column': missing_validation_cols,
    'missing_count': [int(df_clean[col].isna().sum()) for col in missing_validation_cols],
    'empty_string_count': [int((df_clean[col].fillna('').astype(str).str.strip() == '').sum()) for col in missing_validation_cols]
})

missing_validation

,column,missing_count,empty_string_count
0,title,0,0
1,description,0,0
2,content,0,0
3,full_text_clean,0,0


**Nhận xét:** Kiểm chứng missing tập trung vào các cột bắt buộc cho bài toán phân loại. `title`, `description`, `content` phải không còn missing hoặc chuỗi rỗng vì đây là dữ liệu đầu vào chính. Các metadata không dùng trực tiếp làm feature chính chưa cần imputation, vì việc điền giá trị giả cho metadata có thể tạo nhiễu và không phục vụ mục tiêu tạo dataset sạch dùng chung.


### 4.2 Kiểm chứng duplicate

Sau khi deduplicate theo `content`, số duplicate content và số duplicate đồng thời trên `title + description + content` phải bằng 0. Notebook không yêu cầu `title` duplicate hoặc `description` duplicate bằng 0 vì hai trường này có thể trùng tự nhiên.

In [11]:
duplicate_validation = pd.DataFrame({
    'check': [
        'content duplicate after cleaning',
        'title + description + content duplicate after cleaning',
        'title duplicate after cleaning',
        'description duplicate after cleaning'
    ],
    'duplicate_count': [
        int(df_clean.duplicated(subset=['content']).sum()),
        int(df_clean.duplicated(subset=all_text_subset).sum()),
        int(df_clean.duplicated(subset=['title']).sum()),
        int(df_clean.duplicated(subset=['description']).sum())
    ],
    'required_to_be_zero': [
        True,
        True,
        False,
        False
    ]
})

duplicate_validation

,check,duplicate_count,required_to_be_zero
0,content duplicate after cleaning,0,True
1,title + description + content duplicate after cleaning,0,True
2,title duplicate after cleaning,435,False
3,description duplicate after cleaning,880,False


**Nhận xét:** Sau cleaning, duplicate theo `content` và theo bộ `title + description + content` phải bằng 0. Không yêu cầu duplicate theo `title` hoặc `description` riêng lẻ bằng 0 vì các trường này ngắn và có thể trùng tự nhiên.

### 4.3 Kiểm chứng label conflict

Sau xử lý, không còn nhóm `title + description + content` nào có nhiều hơn một `category`.

In [12]:
label_conflict_validation = (
    df_clean.groupby(conflict_subset, dropna=False)['category']
    .nunique()
    .reset_index(name='n_categories')
)

remaining_label_conflict_groups = int((label_conflict_validation['n_categories'] > 1).sum())

pd.DataFrame({
    'metric': ['Remaining label conflict groups'],
    'value': [remaining_label_conflict_groups]
})

,metric,value
0,Remaining label conflict groups,0


**Nhận xét:** Nếu số nhóm label conflict sau xử lý bằng 0, dữ liệu không còn trường hợp cùng một input văn bản nhưng nhiều nhãn khác nhau. Điều này giúp giảm nhiễu nhãn cho bài toán supervised classification.

### 4.4 Kiểm chứng độ dài

Sau xử lý, các bài quá ngắn phải không còn trong dữ liệu. Các bài dài theo ngưỡng P99 vẫn có thể tồn tại vì bài dài có thể hợp lệ; notebook không drop và không tạo cờ riêng cho nhóm này, chỉ thống kê số lượng để kiểm chứng rằng các bài dài vẫn được giữ lại.

In [13]:
length_validation = pd.DataFrame({
    'metric': [
        'Articles with full_text_wc <= 49',
        'Articles with full_text_wc_clean <= 49',
        'Articles with full_text_wc_clean >= 1945'
    ],
    'value': [
        int((df_clean['full_text_wc'] <= 49).sum()),
        int((df_clean['full_text_wc_clean'] <= 49).sum()),
        int((df_clean['full_text_wc_clean'] >= 1945).sum())
    ],
    'required_to_be_zero': [
        True,
        True,
        False
    ]
})

length_validation

,metric,value,required_to_be_zero
0,Articles with full_text_wc <= 49,0,True
1,Articles with full_text_wc_clean <= 49,0,True
2,Articles with full_text_wc_clean >= 1945,2567,False


**Nhận xét:** Kiểm chứng độ dài cần đảm bảo không còn bài quá ngắn theo ngưỡng đã chọn. Các bài có độ dài lớn hơn hoặc bằng P99 vẫn có thể tồn tại và được giữ lại, vì chúng không được xem là lỗi dữ liệu chỉ dựa trên số từ.

### 4.5 Kiểm chứng phân phối nhãn

Nếu phân phối nhãn không thay đổi quá mạnh và không có class nào bị mất hoàn toàn, việc cleaning không làm phá vỡ cấu trúc dữ liệu ban đầu.

In [14]:
category_before = (
    df['category']
    .map(normalize_basic_string)
    .map(lambda x: CATEGORY_MAP.get(remove_accents(x).lower().strip(), x) if not pd.isna(x) else pd.NA)
    .value_counts(dropna=False)
    .reset_index()
)
category_before.columns = ['category', 'count_before']
category_before['percent_before'] = (category_before['count_before'] / len(df) * 100).round(4)

category_after = (
    df_clean['category']
    .value_counts(dropna=False)
    .reset_index()
)
category_after.columns = ['category', 'count_after']
category_after['percent_after'] = (category_after['count_after'] / len(df_clean) * 100).round(4)

category_distribution_compare = category_before.merge(category_after, on='category', how='outer').fillna(0)
category_distribution_compare['count_diff'] = category_distribution_compare['count_after'] - category_distribution_compare['count_before']
category_distribution_compare['percent_diff'] = category_distribution_compare['percent_after'] - category_distribution_compare['percent_before']
category_distribution_compare = category_distribution_compare.sort_values('count_after', ascending=False)

display(category_distribution_compare)

lost_categories = category_distribution_compare.loc[category_distribution_compare['count_after'] == 0, 'category'].tolist()
print(f'Số category bị mất hoàn toàn sau cleaning: {len(lost_categories)}')
print(lost_categories)

,category,count_before,percent_before,count_after,percent_after,count_diff,percent_diff
3,Giải trí,43473,15.9208,41912,15.6319,-1561,-0.2889
9,Thể thao,36271,13.2833,33417,12.4635,-2854,-0.8198
5,Kinh doanh,33348,12.2128,33265,12.4069,-83,0.1941
10,Thời sự,27518,10.0777,27435,10.2324,-83,0.1547
2,Giáo dục,22494,8.2378,22468,8.3799,-26,0.1421
4,Khoa học công nghệ,21045,7.7072,21027,7.8424,-18,0.1352
12,Đời sống,19169,7.0201,19137,7.1375,-32,0.1174
11,Xe,18912,6.9260,18826,7.0215,-86,0.0955
7,Sức khỏe,16697,6.1148,16677,6.2200,-20,0.1052
8,Thế giới,14965,5.4805,14855,5.5405,-110,0.0600


Số category bị mất hoàn toàn sau cleaning: 0
[]


**Nhận xét:** Bảng phân phối nhãn trước và sau cleaning giúp kiểm tra việc làm sạch có làm lệch mạnh cấu trúc category hay không. Điều quan trọng là không có class nào bị mất hoàn toàn và tỷ lệ các lớp không thay đổi bất thường.

### 4.6 Kiểm chứng phân phối source

Source bias chỉ được ghi nhận ở notebook cleaning, không xử lý bằng undersampling vì có thể làm mất dữ liệu hợp lệ. Nếu cần, source bias sẽ được kiểm tra tiếp ở notebook modeling/evaluation.

In [15]:
source_before = df['source'].value_counts(dropna=False).reset_index()
source_before.columns = ['source', 'count_before']
source_before['percent_before'] = (source_before['count_before'] / len(df) * 100).round(4)

source_after = df_clean['source'].value_counts(dropna=False).reset_index()
source_after.columns = ['source', 'count_after']
source_after['percent_after'] = (source_after['count_after'] / len(df_clean) * 100).round(4)

source_distribution_compare = source_before.merge(source_after, on='source', how='outer').fillna(0)
source_distribution_compare['count_diff'] = source_distribution_compare['count_after'] - source_distribution_compare['count_before']
source_distribution_compare['percent_diff'] = source_distribution_compare['percent_after'] - source_distribution_compare['percent_before']
source_distribution_compare

,source,count_before,percent_before,count_after,percent_after,count_diff,percent_diff
0,Thanh Niên,96017,35.1636,95708,35.6962,-309,0.5326
1,Tuổi Trẻ,53010,19.4135,52747,19.6731,-263,0.2596
2,VietnamNet,124031,45.4230,119663,44.6307,-4368,-0.7923


**Nhận xét:** Bảng phân phối source cho biết cleaning có làm thay đổi mạnh tỷ trọng giữa Tuổi Trẻ, Thanh Niên và VietnamNet hay không. Source bias chỉ được ghi nhận ở đây, không xử lý bằng undersampling trong notebook cleaning để tránh làm mất dữ liệu hợp lệ.

### 4.7 Kiểm chứng nhiễu văn bản sau chuẩn hóa

Sau khi chuẩn hóa ba cột lõi và tạo `full_text_clean` để audit, cần kiểm chứng lại xem các ký tự nhiễu đã phát hiện trong EDA như emoji, icon, symbol, ký tự lỗi hiển thị, URL, chú thích ảnh và các dòng điều hướng còn sót lại hay không.

Bước kiểm chứng này không nhằm xử lý thêm dữ liệu, mà dùng để xác nhận rằng hàm chuẩn hóa văn bản đã hoạt động đúng. Nếu vẫn còn nhiều ký tự nhiễu hoặc URL/chú thích ảnh sót lại, cần quay lại điều chỉnh regex trong bước chuẩn hóa văn bản.


In [16]:
detected_noise_chars = '™ → ° ® ● ■ ˚ ` 🡪 ▪ ➢ 🏆 ◊ ✅ ☎ 🌐 📍 ♦ ✔ 🔥 🎥 ℃ 📌 ⚡ ^ 🔹 📺 🗓 ↑ 🎁 ⚽ ▶ ○ 📅 ❖ © 🏟 ´ ㎡ ℅ 📞 🔍 🔴 ✓ ➡ ⏰ 👉 ❌ 🏐 🎯 🥉 🤝 ⚔ 🔑 🔮 🕘 🕒 🔻 ► 📩 � ☑ 🏘 ❤ 📧 📪 ℠ ♥ 📆 ˝ 📄 ㎛ 📝 🏥 🏢 🌷 〒 ￼ ⌘ ⇧ 🔎 🏛 ¸ ⌥ 🔵 🔢 💡 🟢 🧠 🧮 🔝 ⚠ 🕛 🕐 📊 🌟 👑 🥇 ☰ 🔜'

noise_chars = set(detected_noise_chars.split())

remaining_noise = []

for text in df_clean['full_text_clean'].fillna('').astype(str):
    for char in noise_chars:
        if char in text:
            remaining_noise.append(char)

if len(remaining_noise) == 0:
    remaining_noise_summary = pd.DataFrame(columns=['char', 'remaining_count'])
else:
    remaining_noise_summary = (
        pd.Series(remaining_noise)
        .value_counts()
        .reset_index()
    )
    remaining_noise_summary.columns = ['char', 'remaining_count']

remaining_noise_summary

,char,remaining_count


**Nhận xét:** Bảng này kiểm tra các ký tự nhiễu cụ thể đã được thống kê trong EDA. Nếu bảng rỗng hoặc số lượng còn lại rất nhỏ, bước chuẩn hóa ký tự đặc biệt có thể xem là đạt. Nếu còn nhiều ký tự, cần bổ sung vào regex hoặc danh sách `extra_noise_chars`.

In [17]:
noise_patterns = {
    'Remaining URL': r'https?://|www\.',
    'Remaining image caption marker': r'\bảnh\s*:|\bnguồn ảnh\s*:',
    'Remaining xem thêm / đọc thêm': r'\bxem thêm\b|\bđọc thêm\b',
    'Remaining xem chi tiết': r'\bxem chi tiết\b',
    'Remaining replacement character': r'�'
}

noise_pattern_rows = []

for noise_type, pattern in noise_patterns.items():
    mask = df_clean['full_text_clean'].fillna('').str.contains(
        pattern,
        regex=True,
        case=False
    )

    noise_pattern_rows.append({
        'noise_type': noise_type,
        'remaining_count': int(mask.sum())
    })

noise_pattern_check = pd.DataFrame(noise_pattern_rows)
noise_pattern_check

,noise_type,remaining_count
0,Remaining URL,11
1,Remaining image caption marker,2309
2,Remaining xem thêm / đọc thêm,0
3,Remaining xem chi tiết,0
4,Remaining replacement character,0


**Nhận xét:** Bảng này kiểm tra các pattern nhiễu cấp cụm như URL, chú thích ảnh, `Xem thêm`, `Đọc thêm`, `Xem chi tiết` và ký tự lỗi hiển thị. Các giá trị lý tưởng là 0 hoặc rất thấp; nếu còn nhiều dòng sót, cần xem mẫu cụ thể để tránh xóa nhầm nội dung hợp lệ.

## 5. Chia train/dev/test

Do EDA phát hiện class imbalance rõ rệt, khi chia dữ liệu cần dùng stratified split theo `category` để giữ phân phối nhãn tương đối ổn định giữa train, dev và test.

Không xử lý mất cân bằng bằng cách drop dữ liệu ở notebook cleaning vì có thể làm mất dữ liệu hợp lệ. Class imbalance nên xử lý ở notebook modeling bằng macro-F1, class_weight hoặc resampling nếu cần.

Duplicate và label conflict đã được xử lý trước khi split để tránh data leakage. Vectorizer, tokenizer, encoder hoặc các bước học thống kê từ dữ liệu phải được fit chỉ trên train ở notebook modeling sau này.


In [18]:
df_clean = df_clean.reset_index(drop=True)
df_clean['clean_id'] = np.arange(1, len(df_clean) + 1)

train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=42,
    stratify=df_clean['category']
)

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['category']
)

train_df = train_df.copy().reset_index(drop=True)
dev_df = dev_df.copy().reset_index(drop=True)
test_df = test_df.copy().reset_index(drop=True)

split_shape_summary = pd.DataFrame({
    'split': ['train', 'dev', 'test'],
    'n_rows': [len(train_df), len(dev_df), len(test_df)],
    'percent': [
        round(len(train_df) / len(df_clean) * 100, 4),
        round(len(dev_df) / len(df_clean) * 100, 4),
        round(len(test_df) / len(df_clean) * 100, 4)
    ]
})

split_shape_summary


,split,n_rows,percent
0,train,187682,69.9998
1,dev,40218,15.0001
2,test,40218,15.0001


**Nhận xét:** Tập dữ liệu được chia theo tỷ lệ 70/15/15 cho train/dev/test. Vì dùng stratified split theo `category`, ba tập cần giữ được tỷ lệ nhãn tương đối giống nhau. Notebook dùng tên `dev` thay vì lẫn lộn `validation/val` để thống nhất với cách gọi trong báo cáo.


In [19]:
def distribution_table(data, column, split_name):
    temp = data[column].value_counts(dropna=False).reset_index()
    temp.columns = [column, f'{split_name}_count']
    temp[f'{split_name}_percent'] = (temp[f'{split_name}_count'] / len(data) * 100).round(4)
    return temp

train_category_dist = distribution_table(train_df, 'category', 'train')
dev_category_dist = distribution_table(dev_df, 'category', 'dev')
test_category_dist = distribution_table(test_df, 'category', 'test')

category_split_distribution = (
    train_category_dist
    .merge(dev_category_dist, on='category', how='outer')
    .merge(test_category_dist, on='category', how='outer')
    .fillna(0)
)

category_split_distribution


,category,train_count,train_percent,dev_count,dev_percent,test_count,test_percent
0,Bất động sản,4656,2.4808,998,2.4815,997,2.4790
1,Du lịch,6434,3.4281,1379,3.4288,1378,3.4263
2,Giáo dục,15728,8.3801,3370,8.3793,3370,8.3793
3,Giải trí,29338,15.6318,6287,15.6323,6287,15.6323
4,Khoa học công nghệ,14719,7.8425,3154,7.8423,3154,7.8423
5,Kinh doanh,23285,12.4066,4990,12.4074,4990,12.4074
6,Pháp luật,2280,1.2148,488,1.2134,489,1.2159
7,Sức khỏe,11674,6.2201,2501,6.2186,2502,6.2211
8,Thế giới,10398,5.5402,2228,5.5398,2229,5.5423
9,Thể thao,23392,12.4636,5013,12.4646,5012,12.4621


**Nhận xét:** Bảng này kiểm tra phân phối `category` giữa ba tập sau split. Nếu tỷ lệ từng category giữa train, dev và test gần nhau, việc chia dữ liệu không làm lệch phân phối nhãn. Đây là kiểm tra quan trọng vì dataset có mất cân bằng lớp.


In [20]:
train_source_dist = distribution_table(train_df, 'source', 'train')
dev_source_dist = distribution_table(dev_df, 'source', 'dev')
test_source_dist = distribution_table(test_df, 'source', 'test')

source_split_distribution = (
    train_source_dist
    .merge(dev_source_dist, on='source', how='outer')
    .merge(test_source_dist, on='source', how='outer')
    .fillna(0)
)

source_split_distribution


,source,train_count,train_percent,dev_count,dev_percent,test_count,test_percent
0,Thanh Niên,67154,35.7807,14195,35.2951,14359,35.7029
1,Tuổi Trẻ,37033,19.7318,8026,19.9562,7688,19.1158
2,VietnamNet,83495,44.4875,17997,44.7486,18171,45.1813


**Nhận xét:** Bảng này dùng để theo dõi phân phối source trong từng tập. Source không dùng để stratify chính, nhưng cần kiểm tra để tránh trường hợp một nguồn báo bị lệch quá nhiều ở một tập dữ liệu.

In [21]:
train_content_set = set(train_df['content'].dropna())
dev_content_set = set(dev_df['content'].dropna())
test_content_set = set(test_df['content'].dropna())

overlap_summary = pd.DataFrame({
    'pair': [
        'train vs dev',
        'train vs test',
        'dev vs test'
    ],
    'overlap_content_count': [
        len(train_content_set.intersection(dev_content_set)),
        len(train_content_set.intersection(test_content_set)),
        len(dev_content_set.intersection(test_content_set))
    ]
})

overlap_summary


,pair,overlap_content_count
0,train vs dev,0
1,train vs test,0
2,dev vs test,0


**Nhận xét:** Overlap content giữa train, dev và test phải bằng 0. Đây là kiểm tra quan trọng để hạn chế data leakage sau khi đã deduplicate theo `content`. Nếu cùng một nội dung xuất hiện ở nhiều split, mô hình có thể học thuộc bài thay vì học tín hiệu tổng quát.


## 6. Lưu dữ liệu sau xử lý

Dữ liệu sau xử lý được lưu vào thư mục `../data/processed`. File đầu ra chính chỉ chứa bốn cột cốt lõi `title`, `description`, `content`, `category`.

Notebook lưu thêm file mở rộng và file metadata để không làm mất thông tin đã thu thập như `sub_topic`, `tag`, `source`, `url`, `author`, `public_date`. Tuy nhiên, các cột này không được đưa vào file core vì:

- `sub_topic` và `tag` không đồng nhất giữa các báo, có thể thiếu hoặc mang tính chủ quan theo từng website.
- `source`, `url`, `author`, `public_date` chủ yếu phục vụ truy vết và phân tích phụ, không phải trường đầu vào bắt buộc cho bài toán phân loại chủ đề lớn.
- File core cần giữ schema tối giản để người dùng dataset có thể tự quyết định cách kết hợp văn bản và pipeline modeling.


In [ ]:
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

for data in [df_clean, train_df, dev_df, test_df]:
    if 'id' not in data.columns and 'clean_id' in data.columns:
        data.rename(columns={'clean_id': 'id'}, inplace=True)

metadata_cols = [
    'id',
    'category',
    'source',
    'url',
    'author',
    'sub_topic',
    'tag',
    'public_date'
]

model_cols = [
    'id',
    'title',
    'description',
    'content',
    'category'
]

metadata_cols = [col for col in metadata_cols if col in df_clean.columns]
model_cols = [col for col in model_cols if col in df_clean.columns]

news_metadata = df_clean[metadata_cols].copy()

train_save = train_df[model_cols].copy()
dev_save = dev_df[model_cols].copy()
test_save = test_df[model_cols].copy()

news_metadata.to_csv(
    os.path.join(processed_dir, 'news_metadata.csv'),
    index=False,
    encoding='utf-8-sig'
)

train_save.to_csv(
    os.path.join(processed_dir, 'train.csv'),
    index=False,
    encoding='utf-8-sig'
)

dev_save.to_csv(
    os.path.join(processed_dir, 'dev.csv'),
    index=False,
    encoding='utf-8-sig'
)

test_save.to_csv(
    os.path.join(processed_dir, 'test.csv'),
    index=False,
    encoding='utf-8-sig'
)

saved_files_summary = pd.DataFrame({
    'file': [
        'news_metadata.csv',
        'train.csv',
        'dev.csv',
        'test.csv'
    ],
    'role': [
        'Metadata for tracing and optional analysis',
        'Training split for topic classification',
        'Development split for validation/model selection',
        'Test split for final evaluation'
    ],
    'path': [
        os.path.join(processed_dir, 'news_metadata.csv'),
        os.path.join(processed_dir, 'train.csv'),
        os.path.join(processed_dir, 'dev.csv'),
        os.path.join(processed_dir, 'test.csv')
    ],
    'n_rows': [
        len(news_metadata),
        len(train_save),
        len(dev_save),
        len(test_save)
    ],
    'n_cols': [
        news_metadata.shape[1],
        train_save.shape[1],
        dev_save.shape[1],
        test_save.shape[1]
    ],
    'columns': [
        ', '.join(news_metadata.columns),
        ', '.join(train_save.columns),
        ', '.join(dev_save.columns),
        ', '.join(test_save.columns)
    ]
})

saved_files_summary

,file,role,path,n_rows,n_cols,columns
0,news_metadata.csv,Metadata for tracing and optional analysis,/kaggle/working/news_metadata.csv,268118,8,"id, category, source, url, author, sub_topic, tag, public_date"
1,train.csv,Training split for topic classification,/kaggle/working/train.csv,187682,5,"id, title, description, content, category"
2,dev.csv,Development split for validation/model selection,/kaggle/working/dev.csv,40218,5,"id, title, description, content, category"
3,test.csv,Test split for final evaluation,/kaggle/working/test.csv,40218,5,"id, title, description, content, category"


## Lưu dữ liệu sau xử lý

Sau khi làm sạch và chia dữ liệu, notebook chỉ lưu bốn file đầu ra:

- `train.csv`, `dev.csv`, `test.csv`: ba tập dữ liệu chính dùng cho bài toán phân loại chủ đề. Mỗi file gồm các cột `id`, `title`, `description`, `content`, `category`.
- `news_metadata.csv`: file metadata dùng để truy vết và phân tích phụ, gồm `id`, `category`, `source`, `url`, `author`, `sub_topic`, `tag`, `public_date`.

Cách tách này giúp dữ liệu đầu vào cho mô hình được giữ gọn, rõ ràng và không lẫn các trường metadata. Các trường như `source`, `url`, `author`, `sub_topic`, `tag`, `public_date` vẫn được bảo toàn để phục vụ kiểm tra, truy vết hoặc nghiên cứu mở rộng, nhưng không được đưa trực tiếp vào ba file train/dev/test nhằm tránh làm lệch mục tiêu chính của bài toán phân loại chủ đề.

## 7. Kết luận cuối notebook

Notebook đã thực hiện chuẩn hóa kỹ thuật ban đầu trước khi xử lý missing values, duplicate và label conflict. Việc chuẩn hóa sớm giúp hạn chế bỏ sót lỗi do khoảng trắng thừa, xuống dòng, ký tự rỗng, ký tự điều khiển hoặc các biến thể biểu diễn khác nhau của cùng một nội dung.

Các dòng thiếu `title`, `description`, `content` hoặc `category` đã được loại bỏ vì đây là các trường cốt lõi của bài toán phân loại chủ đề tin tức. Notebook không thực hiện điền khuyết cho các trường văn bản này nhằm tránh tạo nội dung giả, làm sai lệch ngữ nghĩa bài viết và gây nhiễu cho các mô hình học máy sau này.

Các nhóm label conflict đã được loại bỏ để tránh tình trạng cùng một nội dung đầu vào nhưng lại được gán nhiều nhãn `category` khác nhau. Duplicate được xử lý dựa trên nội dung văn bản chính, trong đó `content` được xem là trường quan trọng nhất vì chứa phần thông tin đầy đủ nhất của bài báo. Ngoài ra, các trường hợp trùng lặp trên tổ hợp `title`, `description` và `content` cũng được kiểm tra lại nhằm đảm bảo chất lượng dữ liệu sau làm sạch.

Các bài viết quá ngắn theo ngưỡng P01, khoảng 49 từ, đã được loại bỏ vì có khả năng cao là bài bị thiếu nội dung hoặc lỗi trong quá trình thu thập. Ngược lại, các bài quá dài theo ngưỡng P99, khoảng 1.945 từ, không bị loại bỏ tự động vì kiểm tra cho thấy phần lớn vẫn là các bài báo hợp lệ. Các dạng nhiễu phổ biến như URL, chú thích ảnh, dòng điều hướng, emoji, icon, ký tự đặc biệt và khoảng trắng thừa đã được xử lý trong bước chuẩn hóa văn bản.

Notebook tạo bộ dữ liệu sạch ở mức dùng chung bằng cách chuẩn hóa trực tiếp ba cột văn bản `title`, `description` và `content`. Các bước phụ thuộc vào mô hình như lowercase bắt buộc, xóa stopwords, tách từ, tokenize, encode hoặc vectorize không được thực hiện tại notebook này. Cách làm này giúp bộ dữ liệu không bị gắn chặt với một pipeline modeling cụ thể và có thể được sử dụng linh hoạt cho nhiều hướng tiếp cận khác nhau như TF-IDF, SVM, PhoBERT hoặc các mô hình Transformer.

Dữ liệu sau làm sạch được chia thành ba tập `train`, `dev` và `test` bằng stratified split theo `category` để giữ phân phối nhãn tương đối ổn định giữa các tập. Vectorizer, tokenizer hoặc encoder chưa được fit trong notebook này để tránh data leakage; các bước đó cần được thực hiện ở notebook modeling và chỉ được fit trên tập train.

Cuối cùng, notebook chỉ lưu bốn file đầu ra. Ba file `train.csv`, `dev.csv` và `test.csv` là dữ liệu chính dùng cho bài toán phân loại chủ đề, mỗi file gồm các cột `id`, `title`, `description`, `content` và `category`. File `news_metadata.csv` lưu các trường metadata gồm `id`, `category`, `source`, `url`, `author`, `sub_topic`, `tag` và `public_date` để phục vụ truy vết, kiểm tra nguồn dữ liệu hoặc nghiên cứu mở rộng. Cách tách này giúp dữ liệu đầu vào cho mô hình được giữ gọn, rõ ràng và không bị lẫn với các trường metadata phụ.